# OQIM AI - Deep Learning (Neyron Tarmoq) Versiyasi

Ushbu loyihada oddiy Q-Table o'rniga **Deep Q-Network (DQN)** — ya'ni PyTorch neyron tarmoqi yordamida svetoforlarni boshqarish amalga oshiriladi.

Kodlar 5 ta bo'limga ajratilgan. Har bir bo'limni ketma-ket `Shift+Enter` bilan ishga tushiring.

---
## Bo'lim 1: Kutubxonalar va Sozlamalar
Bu yerda loyihaga kerak bo'lgan barcha kutubxonalar yuklanadi va asosiy sozlamalar belgilanadi.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from collections import deque
import random
import copy

# Sozlamalar
EPISODES = 500          # Necha marta o'rganadi
BATCH_SIZE = 64         # Har safar nechta xotiradan o'rganadi
GAMMA = 0.99            # Kelajakni qanchalik muhim deb biladi (0-1)
EPSILON_START = 1.0     # Boshida 100% tasodifiy harakat qiladi
EPSILON_END = 0.01      # Oxirida faqat 1% tasodifiy
EPSILON_DECAY = 0.995   # Har epizodda tasodifiylik shuncha kamayadi
LEARNING_RATE = 0.0005  # Neyron tarmoq o'rganish tezligi
MEMORY_SIZE = 5000      # Xotira hajmi
TARGET_UPDATE = 10      # Har necha epizodda target tarmoq yangilanadi

print("✅ Bo'lim 1 tayyor: Kutubxonalar yuklandi!")
print(f"PyTorch versiyasi: {torch.__version__}")

---
## Bo'lim 2: Chorraha Muhiti (Environment)
Haqiqiy 4 tomonlama chorraha modeli. Shimol-Janub va Sharq-G'arb yo'llari kesishadi. Biri yashil bo'lganda ikkinchisi qizil yonadi.

AI ning vazifasi: ikkala yo'ldan keladigan mashinalarni muvozanatli boshqarish.

In [ ]:
class Intersection:
    """
    4 tomonlama chorraha simulyatori.
    - ns_queue: Shimol-Janub yo'lidagi kutayotgan mashinalar
    - ew_queue: Sharq-G'arb yo'lidagi kutayotgan mashinalar
    """
    def __init__(self):
        self.ns_queue = 0
        self.ew_queue = 0
        self.step_count = 0
        self.max_steps = 60
    
    def get_state(self):
        return np.array([
            self.ns_queue / 40.0,
            self.ew_queue / 40.0,
            (self.step_count % 10) / 10.0
        ], dtype=np.float32)
    
    def step(self, action):
        """
        action=0: NS ga qisqa yashil (15 sek) — 8 ta o'tadi
        action=1: EW ga qisqa yashil (15 sek) — 8 ta o'tadi
        action=2: NS ga uzun yashil (30 sek) — 15 ta o'tadi
        action=3: EW ga uzun yashil (30 sek) — 15 ta o'tadi
        """
        ns_incoming = np.random.poisson(8)
        ew_incoming = np.random.poisson(6)
        self.ns_queue += ns_incoming
        self.ew_queue += ew_incoming
        
        if action == 0:
            ns_pass, ew_pass = 8, 0
        elif action == 1:
            ns_pass, ew_pass = 0, 8
        elif action == 2:
            ns_pass, ew_pass = 15, 0
        else:
            ns_pass, ew_pass = 0, 15
        
        self.ns_queue = max(0, self.ns_queue - ns_pass)
        self.ew_queue = max(0, self.ew_queue - ew_pass)
        self.step_count += 1
        
        # Reward: kamroq mashina kutsa yaxshi, ko'p mashina kutsa yomon
        total = self.ns_queue + self.ew_queue
        reward = 10.0 - total * 0.5  # 0 navbat = +10 baho, 40 navbat = -10 baho
        
        done = self.step_count >= self.max_steps
        return self.get_state(), reward, done
    
    def reset(self):
        self.ns_queue = random.randint(0, 10)
        self.ew_queue = random.randint(0, 10)
        self.step_count = 0
        return self.get_state()

print("✅ Bo'lim 2 tayyor: 4 tomonlama chorraha muhiti yaratildi!")

---
## Bo'lim 3: Neyron Tarmoq (Deep Q-Network)
Bu - loyihaning **miyasi**. 3 ta yashirin qatlamli neyron tarmoq. U chorrahadagi holatni qabul qilib, eng optimal harakatni bashorat qiladi.

Shuningdek, barqaror o'rganish uchun **Target Network** (Ikkinchi nusxa tarmoq) ishlatilgan.

In [ ]:
class DQN(nn.Module):
    """
    Deep Q-Network - Neyron Tarmoq.
    Kirish: 3 ta raqam (NS navbat, EW navbat, vaqt)
    Chiqish: 4 ta harakat uchun Q-qiymatlar
    """
    def __init__(self, state_size=3, action_size=4):
        super(DQN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(state_size, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, action_size)
        )
    
    def forward(self, x):
        return self.network(x)


class ReplayMemory:
    """AI ning xotirasi. O'tgan tajribalarni saqlaydi va ulardan o'rganadi."""
    def __init__(self, capacity):
        self.memory = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        batch = random.sample(self.memory, batch_size)
        states = np.array([b[0] for b in batch])
        actions = [b[1] for b in batch]
        rewards = [b[2] for b in batch]
        next_states = np.array([b[3] for b in batch])
        dones = [b[4] for b in batch]
        return states, actions, rewards, next_states, dones
    
    def __len__(self):
        return len(self.memory)

print("✅ Bo'lim 3 tayyor: Neyron tarmoq arxitekturasi qurildi!")
print(f"Model tuzilishi:\n{DQN()}")

---
## Bo'lim 4: AI ni O'qitish (Training)
Neyron tarmoq 500 marta chorraha boshqarishga urinib ko'radi. Har safar xato qilsa, xatosidan o'rganadi. Natijada baho (Reward) oshib, xatolik (Loss) kamayib boradi.

In [ ]:
# Model va Xotirani yaratish
model = DQN()
target_model = copy.deepcopy(model)  # Target Network (barqaror o'rganish uchun)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
memory = ReplayMemory(MEMORY_SIZE)
env = Intersection()
epsilon = EPSILON_START

reward_history = []
loss_history = []

print("AI o'qitilmoqda... (Bu 15-40 soniya davom etadi)\n")

for episode in range(EPISODES):
    state = env.reset()
    total_reward = 0
    ep_losses = []
    
    while True:
        # Epsilon-Greedy: Boshida ko'p tasodifiy, keyinchalik aqlli
        if random.random() < epsilon:
            action = random.randint(0, 3)
        else:
            with torch.no_grad():
                q_values = model(torch.FloatTensor(state))
                action = q_values.argmax().item()
        
        next_state, reward, done = env.step(action)
        memory.push(state, action, reward, next_state, done)
        total_reward += reward
        state = next_state
        
        # Xotiradan o'rganish (Experience Replay)
        if len(memory) >= BATCH_SIZE:
            states, actions, rewards, next_states, dones = memory.sample(BATCH_SIZE)
            
            states_t = torch.FloatTensor(states)
            actions_t = torch.LongTensor(actions)
            rewards_t = torch.FloatTensor(rewards)
            next_states_t = torch.FloatTensor(next_states)
            dones_t = torch.FloatTensor(dones)
            
            # Hozirgi Q-qiymatlar
            current_q = model(states_t).gather(1, actions_t.unsqueeze(1)).squeeze(1)
            
            # Target Network dan kelajakdagi Q-qiymatlar
            with torch.no_grad():
                next_q = target_model(next_states_t).max(1)[0]
            target_q = rewards_t + GAMMA * next_q * (1 - dones_t)
            
            loss = nn.SmoothL1Loss()(current_q, target_q)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_losses.append(loss.item())
        
        if done:
            break
    
    # Target tarmoqni vaqti-vaqti bilan yangilash
    if (episode + 1) % TARGET_UPDATE == 0:
        target_model.load_state_dict(model.state_dict())
    
    epsilon = max(EPSILON_END, epsilon * EPSILON_DECAY)
    reward_history.append(total_reward)
    if ep_losses:
        loss_history.append(np.mean(ep_losses))
    
    if (episode + 1) % 100 == 0:
        avg = np.mean(reward_history[-50:])
        print(f"Epizod {episode+1}/{EPISODES} | O'rtacha baho: {avg:.1f} | Epsilon: {epsilon:.3f}")

print(f"\n✅ Bo'lim 4 tayyor: AI o'qitildi!")
print(f"Boshlang'ich baho: {np.mean(reward_history[:20]):.1f} -> Yakuniy baho: {np.mean(reward_history[-20:]):.1f}")

---
## Bo'lim 5: Natijalar va Isbot
Bu yerda:
1. AI qanday aqllilashib borgani (Reward ↑, Loss ↓)
2. **Hozirgi holat** vs **AI holat** vs **Kelajak holat** taqqoslanadi
3. DBSCAN bilan tirbandlik zonalari aniqlanadi

In [ ]:
# === A. AI qanday o'rgangani (Training grafiklari) ===
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(reward_history, color='blue', alpha=0.3)
w = 30
smoothed = [np.mean(reward_history[max(0,i-w):i+1]) for i in range(len(reward_history))]
ax1.plot(smoothed, color='blue', linewidth=2.5, label="O'rtacha baho")
ax1.set_title('AI Mukofoti (Reward) — Vaqt o\'tishi bilan oshadi')
ax1.set_xlabel('Epizod')
ax1.set_ylabel('Umumiy Baho')
ax1.legend()
ax1.grid(True)

if loss_history:
    ax2.plot(loss_history, color='red', alpha=0.3)
    sl = [np.mean(loss_history[max(0,i-30):i+1]) for i in range(len(loss_history))]
    ax2.plot(sl, color='red', linewidth=2.5, label="O'rtacha xatolik")
    ax2.set_title('Neyron Tarmoq Xatoligi (Loss) — Vaqt o\'tishi bilan kamayadi')
    ax2.set_xlabel('Epizod')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True)

plt.tight_layout()
plt.show()

# === B. UCH BOSQICHLI TAQQOSLASH ===
test_steps = 80

# 1. Hozirgi holat (fiks svetafor — galma-gal 15 sek)
np.random.seed(42)
random.seed(42)
env_curr = Intersection()
env_curr.reset()
h_current = []
for i in range(test_steps):
    action_fixed = 0 if i % 2 == 0 else 1
    env_curr.step(action_fixed)
    h_current.append(env_curr.ns_queue + env_curr.ew_queue)

# 2. AI holat (DQN)
np.random.seed(42)
random.seed(42)
env_ai = Intersection()
state = env_ai.reset()
h_ai = []
for _ in range(test_steps):
    with torch.no_grad():
        q_vals = model(torch.FloatTensor(state))
        action = q_vals.argmax().item()
    state, _, _ = env_ai.step(action)
    h_ai.append(env_ai.ns_queue + env_ai.ew_queue)

# 3. Kelajak (V2X — mashinalar tezligini moslaydi)
np.random.seed(42)
random.seed(42)
env_fut = Intersection()
state = env_fut.reset()
h_future = []
for _ in range(test_steps):
    with torch.no_grad():
        q_vals = model(torch.FloatTensor(state))
        action = q_vals.argmax().item()
    env_fut.ns_queue = int(env_fut.ns_queue * 0.3)
    env_fut.ew_queue = int(env_fut.ew_queue * 0.3)
    state, _, _ = env_fut.step(action)
    h_future.append(env_fut.ns_queue + env_fut.ew_queue)

plt.figure(figsize=(12, 6))
plt.plot(h_current, label='1. Hozirgi holat (Fiks svetafor)', color='red', linewidth=2.5)
plt.plot(h_ai, label='2. AI joriy etilgandan so\'ng (DQN)', color='blue', linewidth=2.5)
plt.plot(h_future, label='3. Kelajak holati (AI + V2X)', color='green', linewidth=2.5, linestyle='--')
plt.title('Chorrahada Umumiy Kutayotgan Mashinalar (3 Bosqichli Taqqoslash)')
plt.xlabel('Vaqt (Tsikllar)')
plt.ylabel('Kutayotgan Mashinalar Soni')
plt.legend(loc='upper left')
plt.grid(True)
plt.fill_between(range(test_steps), h_current, h_ai, color='red', alpha=0.08)
plt.fill_between(range(test_steps), h_ai, h_future, color='blue', alpha=0.08)
plt.show()

avg_curr = np.mean(h_current)
avg_ai = np.mean(h_ai)
avg_fut = np.mean(h_future)
print(f"\n🔴 Hozirgi holatda o'rtacha kutish: {avg_curr:.1f} ta mashina")
print(f"🔵 AI (DQN) ishlagandan so'ng: {avg_ai:.1f} ta mashina")
print(f"🟢 Kelajak (V2X) tizimida: {avg_fut:.1f} ta mashina")
if avg_curr > 0:
    imp_ai = 100 - (avg_ai / avg_curr * 100)
    imp_fut = 100 - (avg_fut / avg_curr * 100)
    print(f"\n🚀 AI samaradorligi: Tirbandlik {imp_ai:.1f}% ga kamaydi")
    print(f"🚀 Kelajak tizimi: Tirbandlik {imp_fut:.1f}% ga kamaydi")

# === C. DBSCAN ===
np.random.seed(0)
c1 = np.random.normal([30, 30], 5, (100, 2))
c2 = np.random.normal([70, 70], 4, (100, 2))
noise = np.random.uniform(0, 100, (100, 2))
vehicles = np.vstack((c1, c2, noise))
dbscan = DBSCAN(eps=6, min_samples=8)
labels = dbscan.fit_predict(vehicles)

plt.figure(figsize=(8, 6))
plt.scatter(vehicles[labels == -1, 0], vehicles[labels == -1, 1], c='gray', label='Erkin Harakat', alpha=0.5, s=20)
unique = set(labels) - {-1}
clrs = ['red', 'purple', 'orange']
for i, lb in enumerate(unique):
    plt.scatter(vehicles[labels == lb, 0], vehicles[labels == lb, 1], c=clrs[i % 3], label=f'Tirbandlik Zonasi {lb+1}', s=60)
plt.title('DBSCAN: Zichlikka Asoslangan Tirbandlik Tahlili')
plt.legend()
plt.grid(True)
plt.show()

print("\n✅ Barcha natijalar muvaffaqiyatli ko'rsatildi!")